# E3: observed-variant budgets

A negative result with a mechanism.

The expectation was that measuring a plate of multi-mutants would let the
posterior fix real interaction coefficients and pull ahead of pairwise ridge.
The opposite happens. Ridge improves sharply with a budget and the posterior
gets worse.

Produced by `scripts/run_completion_benchmark.py --budget 96 384` and
`scripts/diagnose_budget_conditioning.py`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

pd.set_option("display.width", 180)
ROOT = Path.cwd().parents[1] / "notebooks" / "iclr"


In [2]:
metrics = pd.read_csv(ROOT / "output_260820_e3" / "metrics.csv")
LOWER = {"mae", "rmse", "wasserstein", "crps"}


def compare(reference):
    rows = []
    for arm in ["singles", "budget_96", "budget_384"]:
        block = metrics[metrics.arm == arm]
        for metric in ["spearman", "ndcg", "mae", "rmse"]:
            left = block[(block.model == "posterior") & (block.metric == metric)][["dataset", "value"]]
            right = block[(block.model == reference) & (block.metric == metric)][["dataset", "value"]]
            joined = left.merge(right, on="dataset", suffixes=("_post", "_ref"))
            if joined.empty:
                continue
            better = joined.value_post < joined.value_ref if metric in LOWER else joined.value_post > joined.value_ref
            rows.append({
                "arm": arm,
                "metric": metric,
                "posterior": round(joined.value_post.median(), 3),
                reference: round(joined.value_ref.median(), 3),
                "wins": f"{int(better.sum())}/{len(joined)}",
            })
    return pd.DataFrame(rows).pivot(index="metric", columns="arm")


compare("ridge_additive")

posterior                   ridge_additive                         wins                  
arm      budget_384 budget_96 singles     budget_384 budget_96 singles budget_384 budget_96 singles
metric                                                                                             
mae           0.290     0.320   0.301          0.077     0.141   0.256       1/29      3/29   16/29
ndcg          0.915     0.915   0.921          0.944     0.922   0.889       6/29     10/29   19/29
rmse          0.355     0.383   0.376          0.107     0.176   0.319       0/29      4/29   18/29
spearman      0.132     0.197   0.256          0.404     0.289   0.238       6/29     15/29   21/29

## Against the pairwise baseline

Pairwise ridge is only a distinct model once a budget supplies pair labels.

In [3]:
compare("ridge_pairwise")

posterior                   ridge_pairwise                         wins                  
arm      budget_384 budget_96 singles     budget_384 budget_96 singles budget_384 budget_96 singles
metric                                                                                             
mae           0.290     0.320   0.301          0.085     0.165   0.256       0/29      5/29   16/29
ndcg          0.915     0.915   0.921          0.951     0.922   0.889       3/29     10/29   19/29
rmse          0.355     0.383   0.376          0.121     0.198   0.319       1/29      6/29   18/29
spearman      0.132     0.197   0.256          0.341     0.297   0.238       7/29     15/29   21/29

## Why the budget does not reach the model

A coefficient is fixed only when the variant and every subvariant were measured.
A budget spent on randomly chosen multi-mutants almost never completes a
subvariant set, so most of the measurement never enters the posterior at all.

In [4]:
conditioning = pd.read_csv(ROOT / "output_260820_e3" / "budget_conditioning.csv")
conditioning.groupby("budget")[
    ["observed_combinations", "coefficients_fixed", "usable_fraction"]
].median().round(3)

,observed_combinations,coefficients_fixed,usable_fraction
budget,,,
96,39.0,3.0,0.075
384,327.0,35.0,0.107


Deep libraries fare worst, because a random draw from a 15-mutation library is never a double.

In [5]:
deep = conditioning[conditioning.budget == 96].sort_values("max_depth")
deep[["dataset", "max_depth", "observed_combinations", "coefficients_fixed"]].to_string(index=False)

'             dataset  max_depth  observed_combinations  coefficients_fixed\n               ParD2          3                     38                   4\n            RhlA_RLs          3                     65                  10\n       RhlA_fraction          3                     65                  10\n              TrpB3H          3                     38                   7\n              TrpB3G          3                     38                   7\n              TrpB3F          3                     38                   8\n              TrpB3E          3                     38                   2\n              TrpB3D          3                     39                   4\n              TrpB3I          3                     38                   6\n              TrpB3B          3                     39                   6\n               ParD3          3                     38                   3\n              TrpB3C          3                     38                   6\n           

## What this says about the model

Exact Mobius conditioning is the wrong update rule for a sparse budget. It is
exact where it applies and silent everywhere else. Replacing it with a
regularised regression over all observed variants would let the same budget
inform every coefficient it touches, and that is the next model change.